In [1]:
import sys
import json
import time
import gc
from pathlib import Path
 
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from jiwer import cer
from tqdm.auto import tqdm
import pandas as pd
 
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))
 
from src.data import (
    parse_transcript_file,
    fix_audio_extension,
    ASRDataset,
    build_vocab_from_df,
    make_collate_fn,
)
from src.model_lora import HuBERT_CTC_LoRA
from src.model_houlsby import HuBERT_CTC_Houlsby
from src.train_utils import set_seed, count_parameters, save_checkpoint
from src.eval_utils import run_inference

In [2]:
SSL_MODELS = {
    "mHuBERT-147":  "utter-project/mHuBERT-147-base-3rd-iter",
    "HuBERT-base":  "facebook/hubert-base-ls960",
    "wav2vec2-base": "facebook/wav2vec2-base",
}
 
PEFT_CONFIGS = {
    "lora": {
        "class": "lora",
        "lora_r": 8,
        "lora_alpha": 16,
        "lora_dropout": 0.1,
    },
    "houlsby": {
        "class": "houlsby",
        "adapter_bottleneck": 32,
        "adapter_dropout": 0.1,
    },
}
 
TRAIN_CONFIG = {
    "data_size": "10min",
    "lang": "swa",
    "batch_size": 8,
    "grad_accumulation": 4,
    "learning_rate": 1e-4,
    "weight_decay": 1e-6,
    "target_iterations": 1500,
    "eval_every": 5,
}
 
DATA_DIR = PROJECT_ROOT / "data" / "commonvoice" / "swa"
RESULTS_DIR = PROJECT_ROOT / "results" / "peft"
CHECKPOINTS_DIR = PROJECT_ROOT / "checkpoints" / "peft"
 
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)
 
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

Device: cuda


In [3]:
set_seed(42)
 
df_train = parse_transcript_file(DATA_DIR / "transcript_10min_train.txt")
df_dev = parse_transcript_file(DATA_DIR / "transcript_10min_dev.txt")
df_test = parse_transcript_file(DATA_DIR / "transcript_10min_test.txt")
 
for df in [df_train, df_dev, df_test]:
    fix_audio_extension(df)
 
vocab = build_vocab_from_df(df_train)
idx_to_char = {v: k for k, v in vocab.items()}
collate = make_collate_fn(vocab)
 
AUDIO_DIR = DATA_DIR / "wav"
train_dataset = ASRDataset(df_train, AUDIO_DIR)
dev_dataset = ASRDataset(df_dev, AUDIO_DIR)
test_dataset = ASRDataset(df_test, AUDIO_DIR)
 
train_loader = DataLoader(train_dataset, batch_size=TRAIN_CONFIG["batch_size"],
                          shuffle=True, collate_fn=collate, num_workers=0, pin_memory=True)
dev_loader = DataLoader(dev_dataset, batch_size=TRAIN_CONFIG["batch_size"],
                        shuffle=False, collate_fn=collate, num_workers=0, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=TRAIN_CONFIG["batch_size"],
                         shuffle=False, collate_fn=collate, num_workers=0)
 
effective_batch = TRAIN_CONFIG["batch_size"] * TRAIN_CONFIG["grad_accumulation"]
iterations_per_epoch = len(train_dataset) / effective_batch
num_epochs = int((TRAIN_CONFIG["target_iterations"] / iterations_per_epoch) + 0.9999)
print(f"Vocab size: {len(vocab)} | Train: {len(train_dataset)} | Epochs: {num_epochs}")

Parsed 92 samples from /users/eleves-a/2022/stephane.eilles-chan-way/Desktop/4A/NLP/ml-superb/data/commonvoice/swa/transcript_10min_train.txt
Parsed 103 samples from /users/eleves-a/2022/stephane.eilles-chan-way/Desktop/4A/NLP/ml-superb/data/commonvoice/swa/transcript_10min_dev.txt
Parsed 100 samples from /users/eleves-a/2022/stephane.eilles-chan-way/Desktop/4A/NLP/ml-superb/data/commonvoice/swa/transcript_10min_test.txt
Loaded 92 samples from            utt_id     audio_filename  \
0   cv_swa_000000  cv_swa_000000.wav   
1   cv_swa_000001  cv_swa_000001.wav   
2   cv_swa_000002  cv_swa_000002.wav   
3   cv_swa_000003  cv_swa_000003.wav   
4   cv_swa_000004  cv_swa_000004.wav   
..            ...                ...   
87  cv_swa_000087  cv_swa_000087.wav   
88  cv_swa_000088  cv_swa_000088.wav   
89  cv_swa_000089  cv_swa_000089.wav   
90  cv_swa_000090  cv_swa_000090.wav   
91  cv_swa_000091  cv_swa_000091.wav   

                                                 text  
0   wachambuzi 

In [4]:
def train_epoch_silent(model, loader, criterion, optimizer, device, grad_accum=1):
    """Train one epoch — no per-batch progress bar."""
    model.train()
    total_loss = 0
    optimizer.zero_grad()
 
    for batch_idx, batch in enumerate(loader):
        audios = batch['audios'].to(device)
        texts = batch['texts'].to(device)
        audio_lengths = batch['audio_lengths']
        text_lengths = batch['text_lengths']
 
        logits, T = model(audios, audio_lengths)
        log_probs = F.log_softmax(logits, dim=-1).transpose(0, 1)
        input_lengths = torch.full((log_probs.size(1),), T, dtype=torch.long)
 
        loss = criterion(log_probs, texts, input_lengths, text_lengths)
        loss = loss / grad_accum
        loss.backward()
 
        if (batch_idx + 1) % grad_accum == 0:
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad()
 
        total_loss += loss.item() * grad_accum
 
    if len(loader) % grad_accum != 0:
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        optimizer.zero_grad()
 
    return total_loss / len(loader)
 
 
def evaluate_silent(model, loader, criterion, device):
    """Evaluate — no per-batch progress bar."""
    model.eval()
    total_loss = 0
 
    with torch.no_grad():
        for batch in loader:
            audios = batch['audios'].to(device)
            texts = batch['texts'].to(device)
            audio_lengths = batch['audio_lengths']
            text_lengths = batch['text_lengths']
 
            logits, T = model(audios, audio_lengths)
            log_probs = F.log_softmax(logits, dim=-1).transpose(0, 1)
            input_lengths = torch.full((log_probs.size(1),), T, dtype=torch.long)
 
            loss = criterion(log_probs, texts, input_lengths, text_lengths)
            total_loss += loss.item()
 
    return total_loss / len(loader)

In [5]:
def build_peft_model(ssl_hf_name, peft_name, peft_cfg, vocab_size):
    """Instantiate the right PEFT model."""
    if peft_cfg["class"] == "lora":
        return HuBERT_CTC_LoRA(
            vocab_size=vocab_size,
            model_name=ssl_hf_name,
            lora_r=peft_cfg["lora_r"],
            lora_alpha=peft_cfg["lora_alpha"],
            lora_dropout=peft_cfg["lora_dropout"],
        )
    else:  # houlsby
        return HuBERT_CTC_Houlsby(
            vocab_size=vocab_size,
            model_name=ssl_hf_name,
            adapter_bottleneck=peft_cfg["adapter_bottleneck"],
            adapter_dropout=peft_cfg["adapter_dropout"],
        )

In [6]:
def run_experiment(ssl_name, ssl_hf_name, peft_name, peft_cfg):
    """Full training + evaluation for one (SSL, PEFT) pair."""
    exp_name = f"{ssl_name}_{peft_name}"
    print(f"\n{'='*60}")
    print(f"  Experiment: {exp_name}")
    print(f"  SSL: {ssl_name} ({ssl_hf_name})")
    print(f"  PEFT: {peft_name}")
    print(f"{'='*60}")
 
    set_seed(42)
    start_time = time.time()
 
    # Build model
    model = build_peft_model(ssl_hf_name, peft_name, peft_cfg, len(vocab))
    model = model.to(device)
    total_params, trainable_params = count_parameters(model)
 
    # Optimizer
    criterion = nn.CTCLoss(blank=vocab["<blank>"], zero_infinity=True)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=TRAIN_CONFIG["learning_rate"],
        weight_decay=TRAIN_CONFIG["weight_decay"],
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=10,
    )
 
    # Training loop
    best_dev_loss = float("inf")
    best_ckpt_path = CHECKPOINTS_DIR / f"{exp_name}_best.pt"
    history = {"train_loss": [], "dev_loss": [], "epoch": []}
 
    pbar = tqdm(range(num_epochs), desc=exp_name, unit="epoch")
    for epoch in pbar:
        train_loss = train_epoch_silent(
            model, train_loader, criterion, optimizer,
            device, TRAIN_CONFIG["grad_accumulation"],
        )
 
        if (epoch + 1) % TRAIN_CONFIG["eval_every"] == 0 or epoch == 0 or epoch == num_epochs - 1:
            dev_loss = evaluate_silent(model, dev_loader, criterion, device)
            scheduler.step(dev_loss)
        else:
            dev_loss = history["dev_loss"][-1] if history["dev_loss"] else float("inf")
 
        history["train_loss"].append(train_loss)
        history["dev_loss"].append(dev_loss)
        history["epoch"].append(epoch + 1)
 
        pbar.set_postfix(train=f"{train_loss:.3f}", dev=f"{dev_loss:.3f}")
 
        if dev_loss < best_dev_loss:
            best_dev_loss = dev_loss
            save_checkpoint(
                model=model, optimizer=optimizer, epoch=epoch,
                path=best_ckpt_path, dev_loss=dev_loss,
                config={**TRAIN_CONFIG, "ssl": ssl_name, "peft": peft_name},
                vocab=vocab,
            )
 
    # Test inference
    checkpoint = torch.load(best_ckpt_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()
 
    predictions, references = run_inference(
        model=model, loader=test_loader, device=device,
        idx_to_char=idx_to_char, vocab=vocab,
    )
    test_cer = cer(references, predictions)
    elapsed = (time.time() - start_time) / 60
 
    # Save
    with open(RESULTS_DIR / f"{exp_name}_history.json", "w") as f:
        json.dump(history, f, indent=2)
 
    # Cleanup
    del model, optimizer, scheduler, criterion
    torch.cuda.empty_cache()
    gc.collect()
 
    results = {
        "experiment": exp_name,
        "ssl_model": ssl_name,
        "peft_method": peft_name,
        "total_params_M": round(total_params / 1e6, 1),
        "trainable_params_M": round(trainable_params / 1e6, 2),
        "best_dev_loss": round(best_dev_loss, 4),
        "test_cer": round(test_cer, 4),
        "training_minutes": round(elapsed, 1),
    }
 
    with open(RESULTS_DIR / f"{exp_name}_results.json", "w") as f:
        json.dump(results, f, indent=2)
 
    print(f"  ✓ CER = {test_cer:.4f} | Dev loss = {best_dev_loss:.4f} | {elapsed:.1f} min")
    return results

In [7]:
all_results = []

# load existing results
for json_path in RESULTS_DIR.glob("*_results.json"):
    with open(json_path, "r") as f:
        all_results.append(json.load(f))

# indentify already done experiments
done_experiments = {r["experiment"] for r in all_results}

print(f"Loaded {len(done_experiments)} existing results.")
print("Already done:", sorted(done_experiments))

# run experiments
for ssl_name, ssl_hf_name in SSL_MODELS.items():
    for peft_name, peft_cfg in PEFT_CONFIGS.items():
        exp_name = f"{ssl_name}_{peft_name}"

        if exp_name in done_experiments:
            print(f"Skipping already done: {exp_name}")
            continue

        result = run_experiment(ssl_name, ssl_hf_name, peft_name, peft_cfg)
        all_results.append(result)
        done_experiments.add(exp_name)

Loaded 4 existing results.
Already done: ['HuBERT-base_houlsby', 'HuBERT-base_lora', 'mHuBERT-147_houlsby', 'mHuBERT-147_lora']
Skipping already done: mHuBERT-147_lora
Skipping already done: mHuBERT-147_houlsby
Skipping already done: HuBERT-base_lora
Skipping already done: HuBERT-base_houlsby

  Experiment: wav2vec2-base_lora
  SSL: wav2vec2-base (facebook/wav2vec2-base)
  PEFT: lora


/users/eleves-a/2022/stephane.eilles-chan-way/envs/dl_env/lib64/python3.9/site-packages/transformers/configuration_utils.py:335: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(


Total params: 103.8M
Trainable params: 9.38M (9.0%)
Frozen params: 94.4M


wav2vec2-base_lora:   0%|          | 0/522 [00:00<?, ?epoch/s]

Test inference:   0%|          | 0/13 [00:00<?, ?it/s]

  ✓ CER = 0.3841 | Dev loss = 1.4864 | 36.2 min

  Experiment: wav2vec2-base_houlsby
  SSL: wav2vec2-base (facebook/wav2vec2-base)
  PEFT: houlsby
Total params: 104.7M
Trainable params: 10.29M (9.8%)
Frozen params: 94.4M


wav2vec2-base_houlsby:   0%|          | 0/522 [00:00<?, ?epoch/s]

Test inference:   0%|          | 0/13 [00:00<?, ?it/s]

  ✓ CER = 0.3668 | Dev loss = 1.5115 | 36.3 min


In [8]:
df_results = pd.DataFrame(all_results)
df_results.to_csv(RESULTS_DIR / "summary_peft.csv", index=False)
 
# Pivot: SSL x PEFT → CER
pivot = df_results.pivot(index="ssl_model", columns="peft_method", values="test_cer")
pivot = pivot[["lora", "houlsby"]]
print("\n" + "="*60)
print("  CER by SSL model x PEFT method")
print("="*60)
print(pivot.to_string())
 
# Trainable params
pivot_params = df_results.pivot(index="ssl_model", columns="peft_method", values="trainable_params_M")
pivot_params = pivot_params[["lora", "houlsby"]]
print("\n  Trainable params (M)")
print(pivot_params.to_string())
 
# Ranking per PEFT method
print("\n  SSL ranking per PEFT method (by CER, lower = better):")
for peft in ["lora", "houlsby"]:
    subset = df_results[df_results["peft_method"] == peft].sort_values("test_cer")
    ranking = " > ".join(subset["ssl_model"].tolist())
    print(f"  {peft}: {ranking}")
 
print(f"\nAll results saved to {RESULTS_DIR}/")


  CER by SSL model x PEFT method
peft_method      lora  houlsby
ssl_model                     
HuBERT-base    0.7570   0.3342
mHuBERT-147    0.1868   0.2213
wav2vec2-base  0.3841   0.3668

  Trainable params (M)
peft_method    lora  houlsby
ssl_model                   
HuBERT-base    9.38    10.29
mHuBERT-147    9.38    10.29
wav2vec2-base  9.38    10.29

  SSL ranking per PEFT method (by CER, lower = better):
  lora: mHuBERT-147 > wav2vec2-base > HuBERT-base
  houlsby: mHuBERT-147 > HuBERT-base > wav2vec2-base

All results saved to /users/eleves-a/2022/stephane.eilles-chan-way/Desktop/4A/NLP/ml-superb/results/peft/
